# 🎬 Netflix Recommendation System

## 📌 Objective
Build a content-based recommendation system that suggests similar Netflix titles based on their genres and descriptions.

## 🧠 Approach
We use:
- Text combination between columns
- TF-IDF vectorization
- Cosine similarity

## 📥 Input
A Netflix title

## 📤 Output
Top 5 similar titles

## Imports

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


## 📂 Load Data
We load the cleaned Netflix dataset.

In [2]:
df = pd.read_csv("./data/netflix_cleaned.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0


## 🧹 Data Preparation

We select only relevant columns for recommendation:
- title
- listed_in (genres)
- description

In [3]:
rec_df = df[["title", "listed_in", "description"]].copy()
rec_df.head()

,title,listed_in,description
0,Dick Johnson Is Dead,Documentaries,"As her father nears the end of his life, filmm..."
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,Jailbirds New Orleans,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,Kota Factory,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## 🔧 Feature Engineering

We combine genres and descriptions into a single text feature to represent each title.

In [4]:
rec_df.isnull().sum()

title          0
listed_in      0
description    0
dtype: int64

In [5]:
rec_df["combined_features"] = (
    rec_df["listed_in"] + " " +
    rec_df["description"]
)
rec_df[["title", "combined_features"]].head()

,title,combined_features
0,Dick Johnson Is Dead,Documentaries As her father nears the end of h...
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysterie..."
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act..."
3,Jailbirds New Orleans,"Docuseries, Reality TV Feuds, flirtations and ..."
4,Kota Factory,"International TV Shows, Romantic TV Shows, TV ..."


## 🔢 Text Vectorization (TF-IDF)

We transform the combined text into numerical vectors using TF-IDF.
This allows us to compute similarity between titles.

In [6]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(rec_df["combined_features"])

tfidf_matrix.shape

(8807, 18895)

## 📏 Similarity Computation

We compute cosine similarity between all titles.

In [7]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim.shape

(8807, 8807)

## 🔗 Index Mapping

We map each title to its corresponding index to enable fast lookup.

In [8]:
indices = pd.Series(rec_df.index, index=rec_df["title"])
indices.head()

title
Dick Johnson Is Dead     0
Blood & Water            1
Ganglands                2
Jailbirds New Orleans    3
Kota Factory             4
dtype: int64

## 🤖 Recommendation Function

Given a Netflix title, return the top 5 most similar titles.

In [16]:
def recommend(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]
    movie_indices = [i[0] for i in sim_scores]
    return rec_df["title"].iloc[movie_indices]

## Testing The System

In [17]:
recommend("Narcos")

7463     Miss Dynamite
2921    Narcos: Mexico
6673       El Cartel 2
4750          El Chapo
2            Ganglands
Name: title, dtype: object

In [18]:
recommend("Breaking Bad")

2606                              Extracurricular
4118                                  Iron Ladies
5352    Have You Ever Fallen in Love, Miss Jiang?
4143                                       Sparta
1559                    The Mess You Leave Behind
Name: title, dtype: object

In [19]:
recommend("Ozark")

425                Chicago Med
921                    StartUp
3177            Triad Princess
7463             Miss Dynamite
7325    Lo que la vida me robó
Name: title, dtype: object

In [20]:
recommend("Dark")

869                  Who Killed Sara?
2979                     THE STRANGER
2874                   Altered Carbon
7348                        Love Rain
626     Sophie: A Murder in West Cork
Name: title, dtype: object

## Conclusion

We successfully built a content-based recommendation system using:
- TF-IDF vectorization
- Cosine similarity

### Limitations
- No user personalization
- Depends only on textual similarity

### Future Improvements
- recommendation using embedding for semantic comparaison
- Filter by type (Movie vs TV Show)
- Improve feature weighting